# Tuning the Kalman filter's covariances on DexYCB

Optimises the filter's noise parameters to **minimise angle MAE against ground truth**, then checks the result on sequences it was never fitted on.

**What is actually being tuned**

| parameter | what it controls |
|---|---|
| `jerk_psd` (process noise **Q**) | how freely the hand may change acceleration. Small = smoother, laggier |
| `sigma_angle` (measurement noise **R₁**) | how much the angle measurement is trusted. One value, or one per angle |
| `sigma_xy`, `sigma_z` (**R₂**) | how much MediaPipe's 3D keypoints are trusted, sideways vs in depth |

`r_scale_*` is left at 1: it only multiplies the sigmas, so tuning both would search a flat ridge instead of a point.

**Two rules this notebook follows**
1. The per-angle noise *shape* is estimated from the measured angles themselves (the spread of their frame-to-frame wobble), never from the ground truth. Only one global multiplier is fitted.
2. Sequences are split in half. Parameters are fitted on one half and reported on the other. A gain that appears only on the fitted half is overfitting.

**Before you start:** run `validate_dexycb_colab.ipynb` at least once, so the DexYCB subset and the MediaPipe cache are already in your Drive. This notebook reuses both and never re-runs MediaPipe.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1 · Settings

In [ ]:
KF_DRIVE = '/content/drive/MyDrive/kalman_filtering'
SUBSET_TAR = '/content/drive/MyDrive/dexycb_s01_cam836212060125.tar'
RESULTS_DRIVE = '/content/drive/MyDrive/kalman_filtering_results/dexycb'
MP_CACHE = RESULTS_DRIVE + '/mp_cache'      # written by the validation notebook
SERIAL = '836212060125'

KF = '/content/kalman_filtering'
DATA = '/content/dexycb'
OUT = '/content/results/tuning'

# search effort: 40 + 60 takes roughly 10 min for the angle-only KF,
# 25 min for the EKF. Raise for a finer search.
N_RANDOM, N_REFINE = 40, 60

## 2 · Code, packages and data

In [ ]:
import os, shutil, subprocess, sys

def sh(cmd):
    """Run a shell command and stream its output into the cell."""
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if 'inference_feedback' in line or line.startswith(('W0000', 'I0000')):
            continue
        print(line, end='', flush=True)
    if p.wait():
        raise RuntimeError(f'command failed ({p.returncode}): {cmd}')

assert os.path.isdir(KF_DRIVE), f'upload the kalman_filtering folder to {KF_DRIVE} first'
if os.path.exists(KF):
    shutil.rmtree(KF)
shutil.copytree(KF_DRIVE, KF, ignore=shutil.ignore_patterns('output', '__pycache__', '*.mp4'))

sh('pip -q install mediapipe pyyaml')
os.makedirs(DATA, exist_ok=True)
assert os.path.exists(SUBSET_TAR), 'run validate_dexycb_colab.ipynb first (it builds the subset)'
sh(f'tar -xf "{SUBSET_TAR}" -C {DATA}')
assert os.path.isdir(MP_CACHE), 'MediaPipe cache missing: run the validation notebook first'
print('cached sequences:', len(os.listdir(MP_CACHE)))

## 3 · Load the data once

`prepare()` reads the ground truth and the cached MediaPipe detections, runs the corrector, and builds a hand model per sequence. Everything after this is pure filter maths, so each trial takes a few seconds.

`CALIB = 'mediapipe'` builds the hand model from MediaPipe itself — the realistic live case. Use `'gt'` to tune against the oracle calibration instead.

In [ ]:
import numpy as np
import pandas as pd
sys.path.insert(0, f'{KF}/core')
sys.path.insert(0, f'{KF}/validation')
import validate_dexycb as V
import tune_dexycb as T
from pipeline import AngleCorrector

CALIB = 'mediapipe'
seqs = V.find_sequences(DATA, SERIAL)
corrector = AngleCorrector(f'{KF}/models/angle_corrector_v12.pth')
prep = V.prepare(seqs, f'{KF}/models/hand_landmarker.task', MP_CACHE, corrector, CALIB)
train, test = prep[0::2], prep[1::2]
print(f'{len(prep)} sequences -> {len(train)} train / {len(test)} held out')

### The angle-noise estimate

Per-angle noise, estimated from the corrected angle stream alone. The tuner fits one multiplier on this shape, so angles that are inherently noisier are trusted less, automatically.

In [ ]:
base_sigma = T.estimate_sigma(prep, 'corr')
display(pd.DataFrame({'angle': V.ANGLES, 'sigma_deg': base_sigma.round(2)})
        .sort_values('sigma_deg', ascending=False).head(10))
print(f'median {np.median(base_sigma):.2f} deg, range {base_sigma.min():.2f}-{base_sigma.max():.2f}')

## 4 · Tune the angle-only filter (`kf_angle_corr`)

Two parameters: `jerk_psd` and the noise multiplier.

In [ ]:
sh(f'cd {KF} && python validation/tune_dexycb.py --root {DATA} --serial {SERIAL} '
   f'--cache-dir {MP_CACHE} --out {OUT}/kf_angle_corr --method kf_angle_corr '
   f'--calib {CALIB} --sigma-mode perangle --n-random {N_RANDOM} --n-refine {N_REFINE} --quiet')

## 5 · Tune the full EKF (`ekf_corr`)

Four parameters: the two above plus `sigma_xy` and `sigma_z` for MediaPipe's 3D keypoints. Slower, since every trial runs the 3D measurement update.

In [ ]:
sh(f'cd {KF} && python validation/tune_dexycb.py --root {DATA} --serial {SERIAL} '
   f'--cache-dir {MP_CACHE} --out {OUT}/ekf_corr --method ekf_corr '
   f'--calib {CALIB} --sigma-mode perangle --n-random {N_RANDOM} --n-refine {N_REFINE} --quiet')

## 6 · Results

In [ ]:
import json
from IPython.display import Image, Markdown, display

for name in ('kf_angle_corr', 'ekf_corr'):
    p = f'{OUT}/{name}'
    if not os.path.isdir(p):
        continue
    display(Markdown(open(f'{p}/report.md', encoding='utf-8').read()))
    if os.path.exists(f'{p}/tuning_scatter.png'):
        display(Image(filename=f'{p}/tuning_scatter.png'))
    trials = pd.read_csv(f'{p}/trials.csv')
    display(trials.sort_values('objective').head(5).round(3))

### Reading it
- **Held-out MAE is the only number that counts.** If the tuned setting beats the current default there, it is a real gain.
- The scatter plots show MAE against each parameter. A flat cloud means the filter does not care about that parameter — expected for MAE, which is dominated by the corrector's bias, not by noise.
- Watch `accel err` and `jitter/GT` too: a setting that wins MAE by 0.05° while tripling jitter is not worth taking.

## 7 · Optional: optimise for smoothness as well as accuracy

Pure MAE often prefers a filter that barely smooths. `--accel-weight` adds the acceleration error to the objective, so the search has to keep the output smooth while staying accurate.

In [ ]:
sh(f'cd {KF} && python validation/tune_dexycb.py --root {DATA} --serial {SERIAL} '
   f'--cache-dir {MP_CACHE} --out {OUT}/ekf_corr_smooth --method ekf_corr '
   f'--calib {CALIB} --sigma-mode perangle --accel-weight 0.5 '
   f'--n-random {N_RANDOM} --n-refine {N_REFINE} --quiet')
display(Markdown(open(f'{OUT}/ekf_corr_smooth/report.md', encoding='utf-8').read()))

## 8 · Full comparison with the tuned settings

Re-scores all six methods with the tuned parameters, so you can compare against the untuned table from the validation notebook.

In [ ]:
best = json.load(open(f'{OUT}/ekf_corr/best_config.json'))
cfg = {k: (np.array(v) if isinstance(v, list) else v) for k, v in best['config'].items()}
print('tuned config:', {k: (f'vector({len(v)})' if isinstance(v, np.ndarray) else v) for k, v in cfg.items()})

rows = []
for method in ('mp', 'corr', 'kf_angle_corr', 'ekf_corr'):
    src, filt = V.METHODS[method]
    s_def = T.score(test, {}, src, filt)
    s_tun = T.score(test, cfg, src, filt) if filt else s_def
    rows.append(dict(method=method, mae_default=s_def['mae'], mae_tuned=s_tun['mae'],
                     accel_default=s_def['accel'], accel_tuned=s_tun['accel'],
                     jitter_tuned=s_tun['jitter_vs_gt'], nis1=s_tun['nis1'], nis2=s_tun['nis2']))
display(pd.DataFrame(rows).round(3))

## 9 · Save to Drive and use the tuned settings live

In [ ]:
dest = f'{RESULTS_DRIVE}/tuning'
os.makedirs(dest, exist_ok=True)
shutil.copytree(OUT, dest, dirs_exist_ok=True)
print('saved to', dest)
for name in ('kf_angle_corr', 'ekf_corr', 'ekf_corr_smooth'):
    f = f'{OUT}/{name}/tuned_flags.txt'
    if os.path.exists(f):
        print(f'\n{name}:\n  python run_ekf_corrected.py {open(f).read().strip()}')

On your laptop, copy `tuned_sigma_per_angle.csv` from Drive into the `kalman_filtering` folder and run, for example:

```
python run_ekf_corrected.py --jerk-psd <value> --angle-noise-csv tuned_sigma_per_angle.csv --r-scale-angles 1.0 --sigma-xy <value> --sigma-z <value> --r-scale-kps 1.0
```

The exact line is printed above and stored in `tuned_flags.txt`.